In [21]:
import os
import torch 
import ollama
import re
import tempfile as tem
import subprocess as sp
import inspect
from ollama import  chat,ChatResponse
from pprint import pprint
# from IPython.display import display,Image
# from langgraph.graph import StateGraph,START,END
# from langchain.tools import tool
# from langchain_ollama import ChatOllama
# from langchain.agents import create_agent
# from langchain.messages import HumanMessage,SystemMessage
# from langfuse import Langfuse,observe

In [2]:
Ollama_MODEL="qwen2.5-coder:7b"

In [6]:
stream=chat(model=Ollama_MODEL,
            messages=[{"role":"user","content":"What is your knowledge cut off and how  good are you with verilog ?"}],
            stream=True)
print("Printing the answer\n")
for chunk in stream:
    print(chunk['message']['content'],end='',flush=True)
print()

Printing the answer

As of my last update in October 2023, I have access to a vast amount of information up until that date. However, I don't have real-time internet access or the ability to browse the web after that date.

Regarding Verilog, I can provide general information and help with basic concepts, syntax, and troubleshooting. If you have specific questions about Verilog code or need assistance with a particular aspect of Verilog programming, feel free to ask!


In [14]:
def code_gen(prompt:str)->str:
    stream=chat(model=Ollama_MODEL,messages=[{'role':'System',
                                           "content":"You are a Senoir Design Engineer and well versed with RTL code, provide a clean,synthesizable,well commented,compact Verilog code with respect to the user request."},
                                           {"role":'user',
                                            "content":f"{prompt}"}],
                                            stream=True)
    resp=""
    for chunk in stream:
        content=chunk["message"]["content"]
        print(content,end="",flush=True)
        resp+=content
    return resp

In [ ]:
User_inpt=input("Prompt the verilog moduel requesting")
response=code_gen(User_inpt)

Certainly! Below is a clean, synthesizable, well-commented, and compact Verilog implementation of an APB (Advanced Peripheral Bus) master that completes transactions. This version supports both read and write operations with a 32-bit data width.

```verilog
`timescale 1ns / 1ps

module apb_master (
    // APB Interface Signals
    input wire         clk,          // Clock signal
    input wire         rst_n,        // Active-low reset signal
    
    // APB Master Signals (Write)
    output reg         paddr,        // Address bus
    output reg [31:0]  pwdata,       // Write data bus
    output reg         penable,      // Enable line
    output reg         pwrite,       // Write enable line
    
    // APB Master Signals (Read)
    input wire [31:0]  prdata,       // Read data bus
    input wire         pready,       // Ready line
    input wire         psel,         // Slave select signal
    
    // External Signals
    output reg         pwrite_en     // Write enable for the exter

In [16]:
print(response)

Certainly! Below is a clean, synthesizable, well-commented, and compact Verilog implementation of an APB (Advanced Peripheral Bus) master that completes transactions. This version supports both read and write operations with a 32-bit data width.

```verilog
`timescale 1ns / 1ps

module apb_master (
    // APB Interface Signals
    input wire         clk,          // Clock signal
    input wire         rst_n,        // Active-low reset signal
    
    // APB Master Signals (Write)
    output reg         paddr,        // Address bus
    output reg [31:0]  pwdata,       // Write data bus
    output reg         penable,      // Enable line
    output reg         pwrite,       // Write enable line
    
    // APB Master Signals (Read)
    input wire [31:0]  prdata,       // Read data bus
    input wire         pready,       // Ready line
    input wire         psel,         // Slave select signal
    
    // External Signals
    output reg         pwrite_en     // Write enable for the exter

In [17]:
def code_extraction(code:str)->str:
    pattern=r'module\s+\w+.*?endmodule'
    matches=re.findall(pattern=pattern,string=code,flags=re.DOTALL)
    clean_code=matches[0]
    return clean_code

In [18]:
clean_Vcode=code_extraction(response)
print(clean_Vcode)

module apb_master (
    // APB Interface Signals
    input wire         clk,          // Clock signal
    input wire         rst_n,        // Active-low reset signal
    
    // APB Master Signals (Write)
    output reg         paddr,        // Address bus
    output reg [31:0]  pwdata,       // Write data bus
    output reg         penable,      // Enable line
    output reg         pwrite,       // Write enable line
    
    // APB Master Signals (Read)
    input wire [31:0]  prdata,       // Read data bus
    input wire         pready,       // Ready line
    input wire         psel,         // Slave select signal
    
    // External Signals
    output reg         pwrite_en     // Write enable for the external memory
);

// Internal Registers
reg [31:0]  next_addr;             // Next address to be accessed
reg [31:0]  read_data_reg;           // Register to hold read data

// State Machine States
typedef enum reg [2:0] {
    IDLE,
    WRITE_ADDR, WRITE_DATA, WRITE_WAIT,
    READ_A

In [22]:
def linter(code:str)->dict:
    #temp file creation
     with tem.NamedTemporaryFile(mode="w+t",suffix=".v",delete=False) as temp_file:
          temp_file.write(code)
          temp_file_path=temp_file.name
          try:     
               cmd=["verilator",
                    "--lint-only",
                    "-Wall",
                    "Wno-UNUSED",
                    "Wno-WIDTH",
               temp_file_path]
               out=sp.run(cmd,capture_output=True,text=True,check=False)
               if out.returncode!=0:
                    return{
                         "status":"error",
                         "return_code":out.returncode,
                         "stderr":out.stderr
                    }
               match=re.search(r"\bmodule\s+([A-Za-z_]\w*)",code)
               if not match:
                    return{
                              "status": "error",
                              "returncode": out.returncode,
                              "stderr": "No Verilog module found." 
                         }
               top_module=match.group(1)
               output_file=os.path.join(os.getcwd(),
                                        f"{top_module}.v")
               with open(output_file,"w") as f:
                    f.write(code)
          except sp.CalledProcessError as e:
               return e.stderr
          finally:
               if os.path.exists(temp_file_path):
                    os.remove(temp_file_path)
     return {"stdout":out.stdout,"return_code":out.returncode}

In [23]:
res=linter(response)
print(res)

{'status': 'error', 'return_code': 1, 'stderr': "%Error: Cannot find file containing module: Wno-UNUSED\n%Error: This may be because there's no search path specified with -I<dir>.\n        ... Looked in:\n             Wno-UNUSED\n             Wno-UNUSED.v\n             Wno-UNUSED.sv\n             obj_dir/Wno-UNUSED\n             obj_dir/Wno-UNUSED.v\n             obj_dir/Wno-UNUSED.sv\n%Error: Cannot find file containing module: Wno-WIDTH\n%Error: Exiting due to 3 error(s)\n"}
